In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import os

os.chdir(r"C:\JYOTI MYSQL\Dataset")

print(os.getcwd())

C:\JYOTI MYSQL\Dataset


In [3]:
df = pd.read_csv("Dataset.csv")
print(df.head(10))

   loan_id  customer_id  age  gender marital_status employment_type  \
0   100001       500001   59  Female        Married        Salaried   
1   100002       500002   49    Male        Married        Salaried   
2   100003       500003   35  Female        Married        Salaried   
3   100004       500004   63    Male         Single        Business   
4   100005       500005   28  Female         Single        Salaried   
5   100006       500006   41    Male         Single        Business   
6   100007       500007   59  Female         Single   Self-employed   
7   100008       500008   39    Male         Single        Salaried   
8   100009       500009   43    Male         Single        Business   
9   100010       500010   31    Male        Married        Salaried   

   annual_income    education        city home_ownership  ...  existing_debt  \
0        1508100          PhD  Chandigarh       Mortgage  ...      597530.09   
1         477600   Bachelor's     Lucknow           Rent  

In [4]:
df.shape

(20000, 26)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   loan_id               20000 non-null  int64  
 1   customer_id           20000 non-null  int64  
 2   age                   20000 non-null  int64  
 3   gender                20000 non-null  str    
 4   marital_status        20000 non-null  str    
 5   employment_type       20000 non-null  str    
 6   annual_income         20000 non-null  int64  
 7   education             20000 non-null  str    
 8   city                  20000 non-null  str    
 9   home_ownership        20000 non-null  str    
 10  credit_score          20000 non-null  int64  
 11  loan_amount           20000 non-null  int64  
 12  interest_rate         20000 non-null  float64
 13  loan_term             20000 non-null  int64  
 14  loan_purpose          20000 non-null  str    
 15  monthly_income        20000 no

In [6]:
df.isnull().sum()

loan_id                 0
customer_id             0
age                     0
gender                  0
marital_status          0
employment_type         0
annual_income           0
education               0
city                    0
home_ownership          0
credit_score            0
loan_amount             0
interest_rate           0
loan_term               0
loan_purpose            0
monthly_income          0
existing_debt           0
debt_to_income_ratio    0
employment_years        0
application_date        0
approval_status         0
loan_status             0
default_flag            0
delinquency_days        0
monthly_payment         0
outstanding_amount      0
dtype: int64

In [7]:
print("Duplicate Loan Ids:",df["loan_id"].duplicated().sum())

Duplicate Loan Ids: 0


### Check approval and loan statuses

In [8]:
print("Approval Status:")
display(df["approval_status"].value_counts(dropna=False))

print("Loan Status")
display(df["loan_status"].value_counts(dropna=False))

Approval Status:


approval_status
Rejected    12027
Approved     7973
Name: count, dtype: int64

Loan Status


loan_status
Current    15852
Default     2321
Paid        1827
Name: count, dtype: int64

### Data Quality Summary

In [9]:
quality_summary = pd.DataFrame({
    "Check":[
        "Total Rows",
        "Total Columns",
        "Missing Value",
        "Duplicate Loan IDs",
        "Duplicate Customer IDs",
        "Invalid Age",
        "Invalid Credit Score",
        "Invalid Loan Amount" 
    ],
    "Result":[
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df["loan_id"].duplicated().sum()),
        int(df["customer_id"].duplicated().sum()),
        int((~df["age"].between(18,100)).sum()),
        int((~df["credit_score"].between(300,850)).sum()),
        int((df["loan_amount"] <= 0).sum())
    ]
})
print(quality_summary)  

                    Check  Result
0              Total Rows   20000
1           Total Columns      26
2           Missing Value       0
3      Duplicate Loan IDs       0
4  Duplicate Customer IDs       0
5             Invalid Age       0
6    Invalid Credit Score       0
7     Invalid Loan Amount       0


### Loan Portfolio Overview and Key Performance Indicators (KPIs)

In [10]:
total_applications = len(df)
approved_loans = (df["approval_status"] == "Approved").sum()
rejected_applications = (df["approval_status"] == "Rejected").sum()
approval_rate = (approved_loans / total_applications) * 100
total_approved_loan_amount = df.loc[df["approval_status"] == "Approved", "loan_amount"].sum()
defaulted_loans = ((df["approval_status"] == "Approved") & 
                   (df["loan_status"] == "Default")).sum()
default_rate = (defaulted_loans / approved_loans) * 100 

print("Total Applications:", total_applications)
print("Approved Loans:", approved_loans)
print("Rejected Applications:", rejected_applications)
print(f"Approval Rate: {approval_rate:.2f}%")
print(f"Total Approved Loan Amount: ₹{total_approved_loan_amount:,.0f}")
print("Defaulted Loans:", defaulted_loans)
print(f"Default Rate: {default_rate:.2f}%")

Total Applications: 20000
Approved Loans: 7973
Rejected Applications: 12027
Approval Rate: 39.87%
Total Approved Loan Amount: ₹5,975,289,000
Defaulted Loans: 2321
Default Rate: 29.11%



### Loan Status Distribution Analysis


In [11]:
approved_df = df[df["approval_status"] == "Approved"].copy()

status_counts = approved_df["loan_status"].value_counts()

print(status_counts)

loan_status
Current    3825
Default    2321
Paid       1827
Name: count, dtype: int64


### Calculate the percentage of each status 

In [12]:
status_summary = (
    approved_df["loan_status"].value_counts().reset_index())
status_summary.columns = ["Loan Status", "Number of Loans"]
status_summary["Percentage"] = (
    status_summary["Number of Loans"] / len(approved_df)) * 100
status_summary["Percentage"] = status_summary["Percentage"].round(2)
print(status_summary)

  Loan Status  Number of Loans  Percentage
0     Current             3825       47.97
1     Default             2321       29.11
2        Paid             1827       22.91


### Credit Score and Loan Defaults

In [13]:
# Select approved loans
approved_df = df[df["approval_status"] == "Approved"]

# count approved loans
total_approved = len(approved_df)

#count defaulted loans
total_defaults = (
    approved_df["loan_status"] == "Default").sum()

# calculate default rate 
default_rate = (total_defaults / total_approved) * 100

print("Total Approved Loans:", total_approved)
print("Total Defaulted loans:", total_defaults)
print(f"Overall Default Rate: {default_rate: 2f}%")

print("Average Credit Score:", round(approved_df["credit_score"].mean(),2))
print("Average Credit Score of Defaulted Loans:",
      round(approved_df.loc[approved_df["loan_status"] == "Default","credit_score"].mean(),2))


Total Approved Loans: 7973
Total Defaulted loans: 2321
Overall Default Rate:  29.110749%
Average Credit Score: 719.29
Average Credit Score of Defaulted Loans: 716.06


### Loan Defaults by Age

In [14]:
#select approved loans only
approved_df = df[df["approval_status"] == "Approved"].copy()

#create average age of all approved customers
average_age = approved_df["age"].mean()

#calculate average age of customers whose loans defaulted
defaulted_customers = approved_df[approved_df["loan_status"] == "Default"]
average_default_age = defaulted_customers["age"].mean()

print(round(average_age, 2))
print(round(average_default_age, 2))

42.73
43.05


### Income and Loan Defaults

In [15]:
approved_df = df[df["approval_status"] == "Approved"]

average_income = approved_df["annual_income"].mean()

defaulted_customers = approved_df[approved_df["loan_status"] == "Default"]

average_default_income = defaulted_customers["annual_income"].mean()

print(round(average_income, 2))
print(round(average_default_income, 2))

706670.32
710837.74


### Loan Defaults by Loan Purpose

In [16]:
approved_df = df[df["approval_status"] == "Approved"]

print("Approved Loans by Purpose:")
print(approved_df["loan_purpose"].value_counts())

defaulted_loans = approved_df[approved_df["loan_status"] == "Default"]

print("Defaulted Loans by Purpose:")
print(defaulted_loans["loan_purpose"].value_counts())

Approved Loans by Purpose:
loan_purpose
Business              1177
Medical               1162
Home                  1153
Education             1153
Car                   1112
Personal              1110
Debt Consolidation    1106
Name: count, dtype: int64
Defaulted Loans by Purpose:
loan_purpose
Medical               369
Home                  348
Business              339
Car                   330
Personal              317
Education             310
Debt Consolidation    308
Name: count, dtype: int64


### Validation of Python Analysis Results

In [17]:
print("Total Rows:" , len(df))
print("Unique Loan IDs:" , df["loan_id"].nunique())

print("Approved Loans:",
      (df["approval_status"] == "Approved").sum())
print("Rejected Application:", 
      (df["approval_status"] == "Rejected").sum())

approved_df = df[df["approval_status"] == "Approved"]

print("Defaulted Loans:", 
      (approved_df["loan_status"] == "Default").sum())

print("Current Approved Loans:",
      (approved_df["loan_status"] == "Current").sum())
print("Paid Approved Loans:",
      (approved_df["loan_status"] == "Paid").sum())

print("Rejected Application with current stauts:",
      ((df["approval_status"] == "Rejected") &
       (df["loan_status"] == "Current")).sum())
    



Total Rows: 20000
Unique Loan IDs: 20000
Approved Loans: 7973
Rejected Application: 12027
Defaulted Loans: 2321
Current Approved Loans: 3825
Paid Approved Loans: 1827
Rejected Application with current stauts: 12027
